# Exercise 3: Self-Reflection Prompt for Improving Output

**Tool:** Google Gemini API (`gemini-3.8-flash`) in Google Colab

**Flow:** summarize a text, critique the summary against a rubric, then rewrite it using the critique.

In [ ]:
%pip install -q google-genai

import os
import re
import json
import textwrap
import traceback
import random
from google import genai
from google.genai import types

# Load the API key (Colab secret named GEMINI_API_KEY, or an environment variable locally)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

client = genai.Client()
MODEL = "gemini-3.8-flash"

# Helper: send one prompt to the model and return the reply text
def ask(prompt, system=None):
    config = types.GenerateContentConfig(system_instruction=system)
    return client.models.generate_content(model=MODEL, contents=prompt, config=config).text


In [ ]:
# Source text (an original passage written for this exercise)
article = """Cities are usually several degrees warmer than the countryside around them, a pattern known as the urban heat island effect. The main cause is the materials that make up a city. Asphalt, concrete and dark roofing absorb sunlight during the day and release the stored heat slowly after sunset, so urban areas cool down far less at night than rural ones. Tall buildings add to the problem by blocking breezes and trapping heat between them, while air conditioners, vehicles and industry release additional waste heat directly into the air.

Vegetation works in the opposite direction. Trees provide shade and also cool the air through evapotranspiration, the process by which plants release water vapor. Neighborhoods with little tree cover can be up to ten degrees hotter than leafy ones on the same afternoon, and these hot areas are often the same lower-income districts where fewer residents have air conditioning. Heat waves therefore carry the greatest health risk for people with the fewest resources to cope.

Planners are testing several remedies. Cool roofs painted white or covered in reflective coatings send sunlight back into the sky, green roofs add a layer of plants, and lighter-colored pavements absorb less heat. Planting street trees remains the most popular measure, although young trees take years to provide meaningful shade and need watering to survive. Researchers caution that no single fix is enough: cities see the best results when they combine reflective surfaces, added greenery and targeted investment in the hottest districts first."""

print(len(article.split()), "words in the source text")

In [ ]:
# Prompt 1: summarize
SUMMARY_PROMPT = """Summarize the following text for a general reader with no background in urban planning.

Requirements:
- Exactly 3 sentences and at most 70 words.
- Plain language, no jargon.
- One paragraph, no bullet points or headings.

Text:
{article}"""

# Prompt 2: self-critique against a rubric
CRITIQUE_PROMPT = """You are a strict editor. Critique the summary below against the original text and the requirements.
Score each criterion from 1 to 5 and give one specific, concrete observation for each:
1. Accuracy: any claims that are wrong, exaggerated or not in the original?
2. Coverage: which important points (causes, impacts, solutions) are missing?
3. Clarity: is it easy to follow, with no filler or repetition?
4. Audience: is it plain language suitable for a general reader with no urban planning background?
5. Length: is it exactly 3 sentences and at most 70 words? Count them.
6. Format: is it a single paragraph with no bullet points or headings?
Finish with a bulleted list titled "Fixes to make" with the most important changes.

Original text:
{article}

Summary to critique:
{summary}"""

# Prompt 3: improve the summary using the critique
IMPROVE_PROMPT = """Rewrite the summary so that it applies every item in the critique's "Fixes to make" list.
Requirements: exactly 3 sentences, at most 70 words, plain language for a general reader, one paragraph with no bullet points or headings.
Use only information from the original text.
Return ONLY the improved summary.

Original text:
{article}

Previous summary:
{summary}

Critique:
{critique}"""

In [ ]:
# Step 1: initial summary
summary_v1 = ask(SUMMARY_PROMPT.format(article=article))
print("INITIAL SUMMARY")
print(summary_v1)

# Step 2: the model critiques its own summary
critique = ask(CRITIQUE_PROMPT.format(article=article, summary=summary_v1))
print("\nSELF-CRITIQUE")
print(critique)

# Step 3: improved summary that applies the critique
summary_v2 = ask(IMPROVE_PROMPT.format(article=article, summary=summary_v1, critique=critique))
print("\nIMPROVED SUMMARY")
print(summary_v2)

In [ ]:
# Compare the two versions
sentences_v1 = len(re.findall(r"[.!?](?:\s|$)", summary_v1.strip()))
sentences_v2 = len(re.findall(r"[.!?](?:\s|$)", summary_v2.strip()))
print(f"{'':12}{'Initial':>10}{'Improved':>10}")
print(f"{'Words':12}{len(summary_v1.split()):>10}{len(summary_v2.split()):>10}")
print(f"{'Sentences':12}{sentences_v1:>10}{sentences_v2:>10}")
print(f"{'<= 70 words':12}{str(len(summary_v1.split()) <= 70):>10}{str(len(summary_v2.split()) <= 70):>10}")

# Blind judge: the model picks the better summary without knowing which is which
pair = [("initial", summary_v1), ("improved", summary_v2)]
random.shuffle(pair)
verdict = ask(f"""Which summary is more accurate and complete relative to the text? Answer "A" or "B", then give one sentence of reasoning.

Text:
{article}

Summary A:
{pair[0][1]}

Summary B:
{pair[1][1]}""")
print(f"\nBlind judge (A = {pair[0][0]}, B = {pair[1][0]}):")
print(verdict)

**Reflection:** the rubric made the critique concrete, and the "Fixes to make" list gave the rewrite a checklist to apply. A model can miss its own typical errors, so the notebook also compares word and sentence counts and uses a blind judge with the order shuffled.